In [3]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: D:\dev\dermascan


In [1]:
import torch

In [4]:
from pipeline.pipeline_orchestrator import PipelineOrchestrator
from pipeline.dataclasses.pipeline_config import PipelineConfig, BranchConfig, ModelConfig, TrainerConfig, TrainValTestSplitRatio, AugmentationConfig, CustomSplitConfig
from pipeline.dataclasses.enums import ModelType, FilterType, DatasetLoaderType, AugmentationType, SplitType
from pipeline.dataclasses.training_configs import AdamConfig, ReduceLROnPlateauConfig

ImportError: DLL load failed while importing _C: Das angegebene Modul wurde nicht gefunden.

In [ ]:
pipeline_config = PipelineConfig(
    pipeline_id="one_epoch_full_branch_expirement",
    data_path=str(proj_dir / "data"),
    split_config=CustomSplitConfig(
        train_csv_path=Path(proj_dir / "data" / "custom_split" / "train_df.csv"),
        val_csv_path=Path(proj_dir / "data" / "custom_split" / "val_df.csv"),
        test_csv_path=Path(proj_dir / "data" / "custom_split" / "test_df.csv"),
    ),
    branches=[
        # ──────────────────────────────────────────
        # RGB BRANCHES
        # ──────────────────────────────────────────
        BranchConfig(
            branch_id="rgb_resnet50_finetune_wrs",
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_50,
                filter_type=FilterType.RGB,
                num_classes=2,
                pretrained=True,
                freeze_weights=False,
            ),
            trainer_config=TrainerConfig(
                batch_size=16,
                num_epochs=1,
                learning_rate=0.001,
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig(),
            ),
            dataset_loader=DatasetLoaderType.RGB,
        ),
        BranchConfig(
            branch_id="rgb_resnet101_frozen_wrs_augements",
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_101,
                filter_type=FilterType.RGB,
                num_classes=2,
                pretrained=True,
                freeze_weights=True,
            ),
            trainer_config=TrainerConfig(
                batch_size=8,
                num_epochs=1,
                learning_rate=0.001,
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig(),
            ),
            dataset_loader=DatasetLoaderType.RGB,
            augmentation=AugmentationConfig(
                train=[
                    AugmentationType.RANDOM_HORIZONTAL_FLIP,
                    AugmentationType.RANDOM_VERTICAL_FLIP,
                    AugmentationType.RANDOM_RESIZE_CROP,
                    AugmentationType.COLOR_JITTER,
                ],
                val=[],
            ),
        ),
        # ──────────────────────────────────────────
        # GABOR BRANCHES
        # ──────────────────────────────────────────
        BranchConfig(
            branch_id="gabor_resnet50",
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_50,
                filter_type=FilterType.GABOR,
                num_classes=2,
                pretrained=False,
                freeze_weights=False,
            ),
            trainer_config=TrainerConfig(
                batch_size=16,
                num_epochs=1,
                learning_rate=0.001,
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig(),
            ),
            dataset_loader=DatasetLoaderType.FILTER,
        ),
        # ──────────────────────────────────────────
        # GLCM BRANCHES
        # ──────────────────────────────────────────
        BranchConfig(
            branch_id="glcm_resnet50",
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_50,
                filter_type=FilterType.GLCM,
                num_classes=2,
                pretrained=False,
                freeze_weights=False,
            ),
            trainer_config=TrainerConfig(
                batch_size=16,
                num_epochs=1,
                learning_rate=0.001,
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig(),
            ),
            dataset_loader=DatasetLoaderType.FILTER,
        ),
        # ──────────────────────────────────────────
        # LBP BRANCHES
        # ──────────────────────────────────────────
        BranchConfig(
            branch_id="lbp_resnet50",
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_50,
                filter_type=FilterType.LBP,
                num_classes=2,
                pretrained=False,
                freeze_weights=False,
            ),
            trainer_config=TrainerConfig(
                batch_size=16,
                num_epochs=1,
                learning_rate=0.001,
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig(),
            ),
            dataset_loader=DatasetLoaderType.FILTER,
        ),
    ],
)

In [4]:
# Save the pipeline configuration to a JSON file
with open(proj_dir / "python" / "pipeline" / "doc" / "pipeline_config.json", "w") as f:
    f.write(pipeline_config.model_dump_json(indent=2))

In [5]:
# Load the pipeline configuration from the saved JSON file
json_config = Path(proj_dir / "python" / "pipeline" / "doc" / "pipeline_config.json").read_text()
pipeline_config = PipelineConfig.model_validate_json(json_config)

In [6]:
pipeline_runner = PipelineOrchestrator(
    pipeline_config=pipeline_config, 
    project_root_path=proj_dir
    )

In [ ]:
run = pipeline_runner.run()

In [7]:
run

[BranchResult(branch_id='first_test_branch', training_history=TrainingHistory(train_loss=[0.12791629597081564, 0.09487255454008932], val_loss=[0.10416130184212609, 0.09572485798155256], train_accuracy=[0.853382084095064, 0.8583180987202925], val_accuracy=[0.8576300085251491, 0.8576300085251491], val_f1_macro=[0.4616796695731987, 0.4616796695731987]), metrics_result=MetricsResult(f1_macro=0.4616796695731987, f1_weighted=0.791900677903901, accuracy=0.8576300085251491, precision=0.4288150042625746, sensitivity=0.0, specificity=1.0, auc=0.820960464756372, mcc=0.0))]